# Lab 07: Evaluation-Driven CI/CD

## 🎯 Objectives

Practice release-gate design using synthetic scores; this notebook does not submit baseline or candidate cloud evaluations. Read the [Lab 7 support guide and limitations](README.md) before adapting its pipeline templates.

1. Define synthetic baseline scores
2. Modify agent (simulate change)
3. Define synthetic candidate scores
4. Compare thresholds and regression tolerances
5. Implement gate logic
6. Generate GitHub Actions + Azure DevOps pipelines

## 💡 Key Concept: Evaluation as Release Gate

No deployment without passing evaluation. Types of gates:
- **Absolute**: Score must meet fixed threshold
- **Relative**: No regression from baseline beyond tolerance
- **Critical**: Must pass (block on fail)
- **Advisory**: Warn only (track but don't block)

## ⏱️ Duration: 60-75 minutes

In [1]:
import os
import json
import time
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

env_path = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / '.env.template').is_file()) / '.env'
load_dotenv(env_path)
lab_dir = env_path.parent / 'labs' / 'lab-07-cicd'

project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
tenant_id = os.environ.get("TENANT_ID")
model_deployment = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

print(f"📍 Project: {project_endpoint[:50]}...")
print(f"🤖 Model: {model_deployment}")

📍 Project: https://demopocaifoundry.services.ai.azure.com/api...
🤖 Model: gpt-4.1


In [2]:
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient

credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(endpoint=project_endpoint, credential=credential)
openai_client = project_client.get_openai_client()

print("✅ Clients initialized")

✅ Clients initialized


## Step 1: Define Gate Configuration

The gate configuration defines what must pass before deployment is allowed.

In [ ]:
# Gate configuration - this would live in your repo as gate-config.json
gate_config = {
    "version": "1.0",
    "blocking_criteria": [
        {
            "evaluator": "violence",
            "type": "absolute",
            "threshold": 0,  # Zero defects
            "metric": "defect_count"
        },
        {
            "evaluator": "sexual",
            "type": "absolute",
            "threshold": 0,
            "metric": "defect_count"
        },
        {
            "evaluator": "hate_unfairness",
            "type": "absolute",
            "threshold": 0,
            "metric": "defect_count"
        },
        {
            "evaluator": "groundedness",
            "type": "absolute",
            "threshold": 4.0,
            "metric": "mean_score"
        },
        {
            "evaluator": "task_adherence",
            "type": "relative",
            "max_regression": 0.05,  # No more than 5% drop
            "metric": "mean_score"
        },
        {
            "evaluator": "tool_selection",
            "type": "absolute",
            "threshold": 0.90,
            "metric": "accuracy"
        }
    ],
    "advisory_criteria": [
        {
            "evaluator": "fluency",
            "type": "absolute",
            "threshold": 4.0,
            "metric": "mean_score"
        },
        {
            "evaluator": "coherence",
            "type": "relative",
            "max_regression": 0.10,
            "metric": "mean_score"
        }
    ],
    "minimum_sample_size": 15,
    "confidence_level": 0.95
}

# Save gate config
gate_config_path = lab_dir / 'gate-config.json'
with open(gate_config_path, "w") as f:
    json.dump(gate_config, f, indent=2)

print(f"✅ Gate config saved to {gate_config_path}")
print(f"   Blocking criteria: {len(gate_config['blocking_criteria'])}")
print(f"   Advisory criteria: {len(gate_config['advisory_criteria'])}")

## Step 2: Implement Gate Logic

The gate logic compares baseline vs candidate scores and makes a pass/fail decision.

In [4]:
def evaluate_gate(baseline_scores: dict, candidate_scores: dict, gate_config: dict) -> dict:
    """
    Evaluate release gate criteria.
    
    Returns:
        dict with overall_decision (PASS/FAIL), blocking_results, advisory_results
    """
    blocking_results = []
    advisory_results = []
    overall_pass = True
    
    # Check blocking criteria
    for criteria in gate_config["blocking_criteria"]:
        evaluator = criteria["evaluator"]
        candidate_value = candidate_scores.get(evaluator, {}).get(criteria["metric"])
        
        if candidate_value is None:
            result = {"evaluator": evaluator, "status": "SKIP", "reason": "No data"}
        elif criteria["type"] == "absolute":
            if criteria["metric"] == "defect_count":
                passed = candidate_value <= criteria["threshold"]
            else:
                passed = candidate_value >= criteria["threshold"]
            result = {
                "evaluator": evaluator,
                "status": "PASS" if passed else "FAIL",
                "value": candidate_value,
                "threshold": criteria["threshold"],
                "type": "absolute"
            }
        elif criteria["type"] == "relative":
            baseline_value = baseline_scores.get(evaluator, {}).get(criteria["metric"], 0)
            if baseline_value > 0:
                regression = (baseline_value - candidate_value) / baseline_value
            else:
                regression = 0
            passed = regression <= criteria["max_regression"]
            result = {
                "evaluator": evaluator,
                "status": "PASS" if passed else "FAIL",
                "baseline": baseline_value,
                "candidate": candidate_value,
                "regression": f"{regression:.1%}",
                "max_allowed": f"{criteria['max_regression']:.1%}",
                "type": "relative"
            }
        
        if result["status"] == "FAIL":
            overall_pass = False
        blocking_results.append(result)
    
    # Check advisory criteria (don't affect pass/fail)
    for criteria in gate_config["advisory_criteria"]:
        evaluator = criteria["evaluator"]
        candidate_value = candidate_scores.get(evaluator, {}).get(criteria["metric"])
        baseline_value = baseline_scores.get(evaluator, {}).get(criteria["metric"], 0)
        
        if criteria["type"] == "absolute":
            passed = candidate_value >= criteria["threshold"] if candidate_value else False
        else:
            regression = (baseline_value - (candidate_value or 0)) / baseline_value if baseline_value > 0 else 0
            passed = regression <= criteria["max_regression"]
        
        advisory_results.append({
            "evaluator": evaluator,
            "status": "OK" if passed else "WARN",
            "value": candidate_value,
        })
    
    return {
        "overall_decision": "PASS" if overall_pass else "FAIL",
        "blocking_results": blocking_results,
        "advisory_results": advisory_results,
    }

print("✅ Gate logic function defined")

✅ Gate logic function defined


## Step 3: Simulate Baseline vs Candidate Comparison

In [5]:
# Simulated baseline scores (from Lab 04)
baseline_scores = {
    "violence": {"defect_count": 0, "mean_score": 0.0},
    "sexual": {"defect_count": 0, "mean_score": 0.0},
    "hate_unfairness": {"defect_count": 0, "mean_score": 0.0},
    "groundedness": {"mean_score": 4.3},
    "task_adherence": {"mean_score": 4.2},
    "tool_selection": {"accuracy": 0.92},
    "fluency": {"mean_score": 4.5},
    "coherence": {"mean_score": 4.1},
}

# Simulated candidate scores (after a prompt change)
candidate_scores = {
    "violence": {"defect_count": 0, "mean_score": 0.0},
    "sexual": {"defect_count": 0, "mean_score": 0.0},
    "hate_unfairness": {"defect_count": 0, "mean_score": 0.0},
    "groundedness": {"mean_score": 4.4},  # Improved
    "task_adherence": {"mean_score": 4.1},  # Slight drop but within tolerance
    "tool_selection": {"accuracy": 0.93},  # Improved
    "fluency": {"mean_score": 4.4},  # Slight drop
    "coherence": {"mean_score": 4.2},  # Improved
}

# Run gate evaluation
gate_result = evaluate_gate(baseline_scores, candidate_scores, gate_config)

# Display results
print("="*60)
print("       EVALUATION GATE RESULTS")
print("="*60)
decision_icon = "🟢" if gate_result["overall_decision"] == "PASS" else "🔴"
print(f"\n{decision_icon} Overall Decision: {gate_result['overall_decision']}")

print(f"\n🚦 Blocking Criteria:")
for r in gate_result["blocking_results"]:
    icon = "✅" if r["status"] == "PASS" else "❌"
    if r.get("type") == "absolute":
        print(f"   {icon} {r['evaluator']}: {r.get('value')} (threshold: {r.get('threshold')})")
    elif r.get("type") == "relative":
        print(f"   {icon} {r['evaluator']}: {r.get('candidate')} (regression: {r.get('regression')}, max: {r.get('max_allowed')})")

print(f"\n📋 Advisory Criteria:")
for r in gate_result["advisory_results"]:
    icon = "✅" if r["status"] == "OK" else "⚠️"
    print(f"   {icon} {r['evaluator']}: {r['value']} [{r['status']}]")

       EVALUATION GATE RESULTS

🟢 Overall Decision: PASS

🚦 Blocking Criteria:
   ✅ violence: 0 (threshold: 0)
   ✅ sexual: 0 (threshold: 0)
   ✅ hate_unfairness: 0 (threshold: 0)
   ✅ groundedness: 4.4 (threshold: 4.0)
   ✅ task_adherence: 4.1 (regression: 2.4%, max: 5.0%)
   ✅ tool_selection: 0.93 (threshold: 0.9)

📋 Advisory Criteria:
   ✅ fluency: 4.4 [OK]
   ✅ coherence: 4.2 [OK]


## Step 4: Simulate a Failing Gate

Let's see what happens when a safety evaluator detects a defect.

In [6]:
# Simulated FAILING candidate (safety defect)
failing_candidate = {
    "violence": {"defect_count": 1, "mean_score": 0.5},  # SAFETY DEFECT
    "sexual": {"defect_count": 0, "mean_score": 0.0},
    "hate_unfairness": {"defect_count": 0, "mean_score": 0.0},
    "groundedness": {"mean_score": 3.8},  # Below threshold
    "task_adherence": {"mean_score": 3.5},  # Regression >5%
    "tool_selection": {"accuracy": 0.88},  # Below threshold
    "fluency": {"mean_score": 4.0},
    "coherence": {"mean_score": 3.5},  # Regression
}

failing_result = evaluate_gate(baseline_scores, failing_candidate, gate_config)

print("="*60)
print("       EVALUATION GATE RESULTS (FAILING)")
print("="*60)
print(f"\n🔴 Overall Decision: {failing_result['overall_decision']}")

print(f"\n🚦 Blocking Criteria:")
for r in failing_result["blocking_results"]:
    icon = "✅" if r["status"] == "PASS" else "❌"
    if r.get("type") == "absolute":
        print(f"   {icon} {r['evaluator']}: {r.get('value')} (threshold: {r.get('threshold')})")
    elif r.get("type") == "relative":
        print(f"   {icon} {r['evaluator']}: baseline={r.get('baseline')} → candidate={r.get('candidate')} (regression: {r.get('regression')})")

failures = [r for r in failing_result['blocking_results'] if r['status'] == 'FAIL']
print(f"\n❌ {len(failures)} blocking failures detected — deployment BLOCKED")

       EVALUATION GATE RESULTS (FAILING)

🔴 Overall Decision: FAIL

🚦 Blocking Criteria:
   ❌ violence: 1 (threshold: 0)
   ✅ sexual: 0 (threshold: 0)
   ✅ hate_unfairness: 0 (threshold: 0)
   ❌ groundedness: 3.8 (threshold: 4.0)
   ❌ task_adherence: baseline=4.2 → candidate=3.5 (regression: 16.7%)
   ❌ tool_selection: 0.88 (threshold: 0.9)

❌ 4 blocking failures detected — deployment BLOCKED


## Step 5: Generate GitHub Actions Pipeline

Generate a GitHub Actions adaptation template. Its paths resolve to the support files in this lab, but the cloud integration and metric handling require the changes documented in [the support guide](README.md) before production use.

In [ ]:
# Generate GitHub Actions workflow
github_actions_yaml = """name: AI Agent Evaluation Gate

on:
  pull_request:
    branches: [main]
    paths:
      - 'src/reference-agent/**'
      - 'labs/lab-04-portal-sdk-custom/**'
      - 'labs/lab-07-cicd/**'
      - 'datasets/**'

env:
  AI_FOUNDRY_PROJECT_ENDPOINT: ${{ secrets.AI_FOUNDRY_PROJECT_ENDPOINT }}
  AZURE_AI_MODEL_DEPLOYMENT_NAME: gpt-4o

jobs:
  evaluation-gate:
    runs-on: ubuntu-latest
    permissions:
      contents: read
      pull-requests: write
    
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
          cache: 'pip'
      
      - name: Install dependencies
        run: pip install -r requirements.txt
      
      - name: Azure Login
        uses: azure/login@v2
        with:
          client-id: ${{ secrets.AZURE_CLIENT_ID }}
          tenant-id: ${{ secrets.AZURE_TENANT_ID }}
          subscription-id: ${{ secrets.AZURE_SUBSCRIPTION_ID }}
      
      - name: Run Evaluation
        run: |
          python labs/lab-07-cicd/run_evaluation_gate.py \
            --dataset datasets/golden/contoso-golden-v1.jsonl \
            --gate-config labs/lab-07-cicd/gate-config.json \
            --output-dir ./evaluation-results
      
      - name: Check Gate Decision
        run: |
          DECISION=$(cat ./evaluation-results/gate-decision.json | python -c "import sys,json; print(json.load(sys.stdin)['overall_decision'])")
          echo "Gate decision: $DECISION"
          if [ "$DECISION" = "FAIL" ]; then
            echo "::error::Evaluation gate FAILED. See evaluation report for details."
            exit 1
          fi
      
      - name: Post PR Comment
        if: always()
        uses: actions/github-script@v7
        with:
          script: |
            const fs = require('fs');
            const report = fs.readFileSync('./evaluation-results/pr-summary.md', 'utf8');
            github.rest.issues.createComment({
              issue_number: context.issue.number,
              owner: context.repo.owner,
              repo: context.repo.repo,
              body: report
            });
      
      - name: Upload Evaluation Artifacts
        if: always()
        uses: actions/upload-artifact@v4
        with:
          name: evaluation-results
          path: ./evaluation-results/
"""

# Save GitHub Actions workflow
gh_path = lab_dir / 'github-actions' / 'evaluation-gate.yml'
gh_path.parent.mkdir(parents=True, exist_ok=True)
with open(gh_path, "w") as f:
    f.write(github_actions_yaml)

print(f"✅ GitHub Actions workflow saved to {gh_path}")

In [ ]:
# Generate Azure DevOps pipeline
azure_devops_yaml = """trigger:
  branches:
    include:
      - main
  paths:
    include:
      - src/reference-agent/*
      - labs/lab-04-portal-sdk-custom/*
      - labs/lab-07-cicd/*
      - datasets/*

pr:
  branches:
    include:
      - main

pool:
  vmImage: 'ubuntu-latest'

variables:
  - group: ai-foundry-evaluation  # Variable group with secrets

stages:
  - stage: Evaluation
    displayName: 'AI Agent Evaluation Gate'
    jobs:
      - job: RunEvaluation
        displayName: 'Run Evaluation Suite'
        steps:
          - task: UsePythonVersion@0
            inputs:
              versionSpec: '3.11'
          
          - script: pip install -r requirements.txt
            displayName: 'Install dependencies'
          
          - task: AzureCLI@2
            displayName: 'Run Evaluation Gate'
            inputs:
              azureSubscription: 'ai-foundry-connection'
              scriptType: 'bash'
              scriptLocation: 'inlineScript'
              inlineScript: |
                python labs/lab-07-cicd/run_evaluation_gate.py \
                  --dataset datasets/golden/contoso-golden-v1.jsonl \
                  --gate-config labs/lab-07-cicd/gate-config.json \
                  --output-dir $(Build.ArtifactStagingDirectory)/evaluation-results
          
          - script: |
              DECISION=$(cat $(Build.ArtifactStagingDirectory)/evaluation-results/gate-decision.json | python -c "import sys,json; print(json.load(sys.stdin)['overall_decision'])")
              echo "Gate decision: $DECISION"
              if [ "$DECISION" = "FAIL" ]; then
                echo "##vso[task.logissue type=error]Evaluation gate FAILED"
                exit 1
              fi
            displayName: 'Check Gate Decision'
          
          - task: PublishBuildArtifacts@1
            displayName: 'Publish Evaluation Results'
            condition: always()
            inputs:
              pathtoPublish: '$(Build.ArtifactStagingDirectory)/evaluation-results'
              artifactName: 'evaluation-results'
          
          - task: PublishTestResults@2
            displayName: 'Publish Test Summary'
            condition: always()
            inputs:
              testResultsFormat: 'JUnit'
              testResultsFiles: '$(Build.ArtifactStagingDirectory)/evaluation-results/junit-results.xml'
"""

# Save Azure DevOps pipeline
ado_path = lab_dir / 'azure-devops' / 'evaluation-gate.yml'
ado_path.parent.mkdir(parents=True, exist_ok=True)
with open(ado_path, "w") as f:
    f.write(azure_devops_yaml)

print(f"✅ Azure DevOps pipeline saved to {ado_path}")

## Step 6: Generate PR Summary Report

Create a markdown report that gets posted to PRs with evaluation results.

In [9]:
def generate_pr_summary(gate_result: dict, baseline_scores: dict, candidate_scores: dict) -> str:
    """Generate a markdown PR summary for evaluation results."""
    decision = gate_result["overall_decision"]
    icon = "✅" if decision == "PASS" else "❌"
    
    summary = f"""## {icon} Evaluation Gate: {decision}

### Blocking Criteria

| Evaluator | Status | Baseline | Candidate | Threshold |
|-----------|--------|----------|-----------|----------|
"""
    for r in gate_result["blocking_results"]:
        status_icon = "✅" if r["status"] == "PASS" else "❌"
        baseline_val = r.get("baseline", "-")
        candidate_val = r.get("value", r.get("candidate", "-"))
        threshold = r.get("threshold", r.get("max_allowed", "-"))
        summary += f"| {r['evaluator']} | {status_icon} {r['status']} | {baseline_val} | {candidate_val} | {threshold} |\n"
    
    summary += "\n### Advisory Criteria\n\n"
    summary += "| Evaluator | Status | Value |\n|-----------|--------|-------|\n"
    for r in gate_result["advisory_results"]:
        status_icon = "✅" if r["status"] == "OK" else "⚠️"
        summary += f"| {r['evaluator']} | {status_icon} {r['status']} | {r['value']} |\n"
    
    if decision == "FAIL":
        failures = [r for r in gate_result["blocking_results"] if r["status"] == "FAIL"]
        summary += f"\n### ❌ {len(failures)} Blocking Failure(s)\n\n"
        for f in failures:
            summary += f"- **{f['evaluator']}**: "
            if f.get("type") == "absolute":
                summary += f"Score {f.get('value')} below threshold {f.get('threshold')}\n"
            else:
                summary += f"Regression {f.get('regression')} exceeds max {f.get('max_allowed')}\n"
    
    return summary

# Generate and display PR summary
pr_summary = generate_pr_summary(gate_result, baseline_scores, candidate_scores)
print(pr_summary)

# Also show failing case
print("\n" + "="*60 + "\n")
failing_summary = generate_pr_summary(failing_result, baseline_scores, failing_candidate)
print(failing_summary)

## ✅ Evaluation Gate: PASS

### Blocking Criteria

| Evaluator | Status | Baseline | Candidate | Threshold |
|-----------|--------|----------|-----------|----------|
| violence | ✅ PASS | - | 0 | 0 |
| sexual | ✅ PASS | - | 0 | 0 |
| hate_unfairness | ✅ PASS | - | 0 | 0 |
| groundedness | ✅ PASS | - | 4.4 | 4.0 |
| task_adherence | ✅ PASS | 4.2 | 4.1 | 5.0% |
| tool_selection | ✅ PASS | - | 0.93 | 0.9 |

### Advisory Criteria

| Evaluator | Status | Value |
|-----------|--------|-------|
| fluency | ✅ OK | 4.4 |
| coherence | ✅ OK | 4.2 |



## ❌ Evaluation Gate: FAIL

### Blocking Criteria

| Evaluator | Status | Baseline | Candidate | Threshold |
|-----------|--------|----------|-----------|----------|
| violence | ❌ FAIL | - | 1 | 0 |
| sexual | ✅ PASS | - | 0 | 0 |
| hate_unfairness | ✅ PASS | - | 0 | 0 |
| groundedness | ❌ FAIL | - | 3.8 | 4.0 |
| task_adherence | ❌ FAIL | 4.2 | 3.5 | 5.0% |
| tool_selection | ❌ FAIL | - | 0.88 | 0.9 |

### Advisory Criteria

| Evaluator | Status 

## ✅ Lab 07 Complete

### What You Created
- **Gate configuration** (JSON) defining blocking + advisory criteria
- **Gate logic** (Python) for comparing baseline vs candidate
- **GitHub Actions pipeline** with evaluation gate
- **Azure DevOps pipeline** with evaluation gate
- **PR summary generator** for automated reporting

### Lab Support Files
The runner and configuration are retained with this lab. The notebook generates the two YAML files shown below; they are adaptation templates, not validated production pipelines:
```
labs/lab-07-cicd/
├── gate-config.json              # What to evaluate
├── run_evaluation_gate.py        # Orchestration script
├── github-actions/evaluation-gate.yml
└── azure-devops/evaluation-gate.yml
```

### Next Steps
- **Lab 08**: Extend to production monitoring
- Follow the [support guide](README.md) before adapting a pipeline to your repository; do not compare the synthetic examples to Lab 4 cloud scores.